# Multi-CSV Validation Notebook


In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
%matplotlib inline
from tqdm.auto import tqdm
tqdm.pandas()
import ast
import metric


In [ ]:
# Load ground truth data
df = pd.read_csv("train_set.csv")
df = df.sample(n=5, random_state=33).reset_index(drop=True)
df_parsed = df.copy()
df['multiple_choice_qa'] = df_parsed['multiple_choice_qa'].apply(ast.literal_eval)
print(f"Loaded ground truth data: {len(df)} samples")
df


In [ ]:
# Find all CSV result files
csv_files = glob.glob("res_*.csv")
print(f"Found {len(csv_files)} CSV files:")
for file in csv_files:
    print(f"  - {file}")

if not csv_files:
    print("❌ No CSV files found!")
else:
    print(f"\n📊 Will evaluate {len(csv_files)} files")


In [ ]:
# Function to evaluate a single CSV file
def evaluate_csv_file(csv_file):
    print(f"\n{'='*60}")
    print(f"Evaluating: {csv_file}")
    print(f"{'='*60}")
    
    try:
        # Load results
        df_results = pd.read_csv(csv_file)
        print(f"Loaded {len(df_results)} results from {csv_file}")
        
        # Merge with ground truth
        df_eval = df.copy()
        df_eval['svg'] = df_results['svg']
        
        # Calculate scores
        df_eval['svg_score'] = df_eval.progress_apply(
            lambda r: metric.score_instance(r.multiple_choice_qa, r.svg, random_seed=42),
            axis=1,
        )
        
        # Calculate mean scores
        mean_svg_score = pd.DataFrame(df_eval['svg_score'].tolist()).mean(axis=0)
        
        print('='*20)
        print(mean_svg_score)
        print()
        print(f'Final svg score: {mean_svg_score.competition_score}')
        
        return {
            'file': csv_file,
            'samples': len(df_results),
            'vqa': mean_svg_score.vqa_score,
            'aesthetic': mean_svg_score.aesthetic_score,
            'ocr': mean_svg_score.ocr_score,
            'competition': mean_svg_score.competition_score,
            'df_eval': df_eval
        }
        
    except Exception as e:
        print(f"❌ Error evaluating {csv_file}: {e}")
        return None


In [ ]:
# Evaluate all CSV files
print("🚀 Starting evaluation of all CSV files...")
print("="*80)

all_results = []
for csv_file in csv_files:
    result = evaluate_csv_file(csv_file)
    if result:
        all_results.append(result)

print(f"\n🎯 Evaluation completed for {len(all_results)} files")


In [ ]:
# Create comparison table
if all_results:
    print("\n" + "="*80)
    print("📊 COMPARISON TABLE")
    print("="*80)
    
    # Create DataFrame for comparison
    comparison_data = []
    for result in all_results:
        comparison_data.append({
            'File': result['file'].replace('.csv', ''),
            'Samples': result['samples'],
            'VQA': f"{result['vqa']:.4f}",
            'Aesthetic': f"{result['aesthetic']:.4f}",
            'OCR': f"{result['ocr']:.4f}",
            'Competition': f"{result['competition']:.4f}"
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Sort by Competition Score (highest first)
    comparison_df = comparison_df.sort_values('Competition', ascending=False)
    
    print(comparison_df.to_string(index=False))
    
    # Find best performing model
    best_model = comparison_df.iloc[0]
    print(f"\n🏆 BEST PERFORMING MODEL: {best_model['File']}")
    print(f"   Competition Score: {best_model['Competition']}")
    print(f"   VQA: {best_model['VQA']}, Aesthetic: {best_model['Aesthetic']}, OCR: {best_model['OCR']}")
    
else:
    print("❌ No results to compare!")


In [ ]:
# Visualize ALL models side by side for each prompt
if all_results:
    print(f"\n🎨 Visualizing ALL models side by side for each prompt")
    print(f"Found {len(all_results)} models to compare")
    
    # Get the number of samples (assuming all have same number)
    num_samples = len(all_results[0]['df_eval'])
    
    for sample_idx in range(num_samples):
        print(f"\n{'='*80}")
        print(f"PROMPT {sample_idx + 1}: {all_results[0]['df_eval'].iloc[sample_idx].description}")
        print(f"{'='*80}")
        
        # Create subplot for all models
        fig, axes = plt.subplots(1, len(all_results), figsize=(6 * len(all_results), 4))
        if len(all_results) == 1:
            axes = [axes]  # Make it iterable for single model
        
        for model_idx, result in enumerate(all_results):
            df_eval = result['df_eval']
            r = df_eval.iloc[sample_idx]
            
            s_vqa = r.svg_score['vqa_score']
            s_aesthetic = r.svg_score['aesthetic_score']
            s_ocr = r.svg_score['ocr_score']
            s_score = r.svg_score['competition_score']
            
            # Plot SVG
            axes[model_idx].imshow(metric.svg_to_png(r.svg))
            axes[model_idx].axis('off')
            axes[model_idx].set_title(f'{result["file"].replace(".csv", "")}\n'
                                    f'score={s_score:.2f}, vqa={s_vqa:.2f}\n'
                                    f'ocr={s_ocr:.2f}, aes={s_aesthetic:.2f}', 
                                    fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
else:
    print("No results to visualize!")
